# 结构化输出

让 Agent 按指定格式返回结构化数据，而非自由文本。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## 传入 Pydantic 模型

将 Pydantic 模型传给 `response_format` 参数。

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import json

class CourseInfo(BaseModel):
    """课程信息"""
    course_name: str = Field(description="课程名称")
    difficulty: str = Field(description="难度")
    estimated_hours: int = Field(description="预计时长")
    is_free: bool = Field(description="是否免费")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# 将 Pydantic 模型作为工具绑定
model_with_tools = model.bind_tools([CourseInfo])

# 传入非结构化文本，tool_call 参数即结构化数据
text = "我在学 Python3 基础教程，入门级别，20小时，完全免费"
response = model_with_tools.invoke(text)

if response.tool_calls:
    data = response.tool_calls[0]["args"]
    print(f"课程: {data['course_name']}")
    print(f"难度: {data['difficulty']}")
    print(f"时长: {data['estimated_hours']}小时")
    print(f"免费: {data['is_free']}")
else:
    print(f"模型回复: {response.content}")

## 与工具共存

`response_format` 和 `tools` 可同时使用。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
def search_course(keyword: str) -> str:
    courses = {"python": "Python3 基础教程 | 免费"}
    return courses.get(keyword.lower(), "未找到")

class CourseRecommendation(BaseModel):
    course_name: str = Field(description="推荐课程")
    reason: str = Field(description="理由")
    difficulty: str = Field(description="难度")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[search_course], response_format=CourseRecommendation, system_prompt="先查询再推荐。")
print("工具+结构化输出 Agent 已创建")

## 复杂嵌套结构

支持嵌套、列表、枚举。

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Topic(BaseModel):
    name: str = Field(description="知识点")
    order: int = Field(description="顺序")
    minutes: int = Field(description="分钟")

class LearningPlan(BaseModel):
    goal: str = Field(description="目标")
    level: Literal["入门", "进阶", "高级"] = Field(description="难度")
    total_hours: float = Field(description="总时长")
    topics: list[Topic] = Field(description="知识点列表")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, response_format=LearningPlan, system_prompt="你是学习规划师。")
print("嵌套结构化输出 Agent 已创建")

## 直接用 with_structured_output()

不需要 Agent 时，直接用模型的方法。

In [ ]:
class SentimentResult(BaseModel):
    """情感分析"""
    sentiment: str = Field(description="积极/消极/中性")
    score: float = Field(description="强度 0~1")
    keywords: list[str] = Field(description="关键情感词")

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)

# 使用 bind_tools() 替代 with_structured_output()
model_with_tools = model.bind_tools([SentimentResult])
# texts = [
#     "菜鸟教程 RUNOOB 真的太好用了！",
#     "这个教程内容太少了。",
# ]
# for text in texts:
#     response = model_with_tools.invoke(text)
#     if response.tool_calls:
#         data = response.tool_calls[0]["args"]
#         print(f"情感: {data['sentiment']}, 强度: {data['score']}")

print("bind_tools 模型已创建，取消注释 invoke 测试")

**术语：** response_format、structured_response、with_structured_output()、Pydantic BaseModel